# Data quality metrics

Runs the same engine as `metrics.py` against a live CluedIn instance and
shows the result, without writing anything to SQL Server. Use it to try a
check out before adding it to the configuration file.


In [ ]:
# The notebooks run from the repository root so that the modules import.
import os
import sys
import warnings

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

# Every input comes from the metrics configuration -- no hard-coded entity
# types or vocabulary keys. Point METRICS_CONFIG at your own file.
CONFIG_PATH = os.getenv('METRICS_CONFIG', 'config/metrics.example.json')
CONFIG_PATH


In [ ]:
from config import load_config
from api import get_entities_by_type
from entity_metrics import (build_reference_sets,
                            calculate_dynamic_metrics,
                            calculate_entity_type_metrics)

config = load_config(CONFIG_PATH)
print(config.title)
print(f'{len(config.entity_types)} entity type(s), '
      f'{sum(len(e.checks) for e in config.entity_types)} check(s)')


## Load every entity type the configuration refers to


In [ ]:
entities_by_type = {
    entity_type: get_entities_by_type(entity_type,
                                      page_size=config.page_size)
    for entity_type in config.entity_types_to_load()
}

{k: len(v) for k, v in entities_by_type.items()}


## Calculate the metrics


In [ ]:
reference_sets = build_reference_sets(config, entities_by_type)

records = []
for entity_type_config in config.entity_types:
    records += calculate_entity_type_metrics(
        config=config,
        entity_type_config=entity_type_config,
        entities=entities_by_type[entity_type_config.entity_type],
        reference_sets=reference_sets)

for entity_type in config.dynamic_metric_entity_types():
    records += calculate_dynamic_metrics(
        config=config,
        entity_type=entity_type,
        entities=entities_by_type[entity_type])

print(f'{len(records)} metrics')


## The worst metrics first


In [ ]:
for record in sorted(records, key=lambda r: r.value):
    print(f'{record.value:7.1%}  {record.name}  ({record.info})')


## Chart them by dimension


In [ ]:
%pip install matplotlib

import matplotlib.pyplot as plt

DIMENSION = 'Completeness'

selected = sorted(
    [r for r in records if r.name.startswith(DIMENSION)],
    key=lambda r: r.value)

if selected:
    plt.figure(figsize=(10, max(3, len(selected) * 0.3)))
    plt.barh([r.name[len(DIMENSION) + 4:] for r in selected],
             [r.value for r in selected])
    plt.xlim(0, 1)
    plt.title(f'{DIMENSION} ({len(selected)} metrics)')
    plt.tight_layout()
    plt.show()
else:
    print(f'No {DIMENSION} metrics configured.')
